# Softmax Activation Function

## Overview
The softmax function is an activation function commonly used in the output layer of multi-class classification neural networks. It converts raw model scores (logits) into a probability distribution over multiple classes.

## Key Concepts
- **Softmax Formula**: Converts a vector of K real numbers into a probability distribution of K possible outcomes
- **Properties**: Output sums to 1, values between 0 and 1, differentiable
- **Use Case**: Multi-class classification where each sample belongs to exactly one class
- **Relationship**: Generalization of sigmoid for multiple classes

## Import Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import softmax as scipy_softmax

# Set random seed for reproducibility
np.random.seed(42)

## Softmax Formula

The softmax function for class j is defined as:

$$\sigma(z)_j = \frac{e^{z_j}}{\sum_{k=1}^{K} e^{z_k}}$$

Where:
- $z_j$ is the raw score (logit) for class j
- $K$ is the total number of classes
- The denominator is the sum of exponentials of all scores

## Implement Softmax Function

In [ ]:
def softmax(z):
    """
    Compute softmax for each row of z.
    
    Args:
        z: Input array of shape (n_samples, n_classes) or 1D array
    
    Returns:
        Softmax probabilities of the same shape as input
    """
    # Handle numerical stability: subtract max for numerical stability
    z_shifted = z - np.max(z, axis=-1, keepdims=True)
    
    # Compute exponentials
    exp_z = np.exp(z_shifted)
    
    # Normalize by sum of exponentials
    return exp_z / np.sum(exp_z, axis=-1, keepdims=True)

# Test with a single sample
logits = np.array([2.0, 1.0, 0.1])
probs = softmax(logits)
print("Logits:", logits)
print("Probabilities:", probs)
print("Sum of probabilities:", np.sum(probs))

## Softmax for Batch of Samples

In [ ]:
# Example with multiple samples (batch)
batch_logits = np.array([
    [3.0, 1.0, 0.2],  # Sample 1: likely class 0
    [0.1, 2.0, 0.3],  # Sample 2: likely class 1
    [0.2, 0.1, 3.0]   # Sample 3: likely class 2
])

batch_probs = softmax(batch_logits)
print("Batch Logits:")
print(batch_logits)
print("\nBatch Probabilities:")
print(batch_probs)
print("\nSum per sample (should be 1):", np.sum(batch_probs, axis=1))

## Compare with SciPy Implementation

In [ ]:
# Verify our implementation matches scipy
scipy_probs = scipy_softmax(batch_logits, axis=1)
print("Our implementation:")
print(batch_probs)
print("\nSciPy implementation:")
print(scipy_probs)
print("\nDifference:", np.max(np.abs(batch_probs - scipy_probs)))

## Properties of Softmax

In [ ]:
print("Property 1: Output is a probability distribution")
print(f"  Min probability: {np.min(batch_probs):.6f}")
print(f"  Max probability: {np.max(batch_probs):.6f}")
print(f"  All values between 0 and 1: {np.all((batch_probs >= 0) & (batch_probs <= 1))}")

print("\nProperty 2: Probabilities sum to 1 for each sample")
print(f"  Row sums: {np.sum(batch_probs, axis=1)}")

print("\nProperty 3: Softmax preserves ordering")
sample_idx = 0
print(f"  Logits order: {np.argsort(batch_logits[sample_idx])[::-1]}")
print(f"  Probs order:  {np.argsort(batch_probs[sample_idx])[::-1]}")

## Visualize Softmax Output

In [ ]:
# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Logits vs Probabilities for 3 classes
classes = ['Class 0', 'Class 1', 'Class 2']
x = np.arange(len(classes))
width = 0.35

# Normalize logits for comparison
logits_sample = batch_logits[0]
probs_sample = batch_probs[0]

axes[0].bar(x - width/2, logits_sample, width, label='Raw Logits', alpha=0.8)
axes[0].bar(x + width/2, probs_sample, width, label='Softmax Probs', alpha=0.8)
axes[0].set_xlabel('Classes')
axes[0].set_ylabel('Value')
axes[0].set_title('Raw Logits vs Softmax Probabilities (Sample 1)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(classes)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Heatmap of all batch probabilities
im = axes[1].imshow(batch_probs, cmap='YlOrRd', aspect='auto')
axes[1].set_xlabel('Classes')
axes[1].set_ylabel('Samples')
axes[1].set_title('Softmax Probabilities Heatmap')
axes[1].set_xticks(np.arange(batch_probs.shape[1]))
axes[1].set_yticks(np.arange(batch_probs.shape[0]))
axes[1].set_yticklabels([f'Sample {i}' for i in range(batch_probs.shape[0])])
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

print("\nProbability values:")
for i, row in enumerate(batch_probs):
    print(f"Sample {i}: {row} -> Predicted class: {np.argmax(row)}")

## Derivative of Softmax (for backpropagation)

In [ ]:
def softmax_jacobian(z):
    """
    Compute Jacobian matrix of softmax (derivative).
    
    For softmax output s_i = exp(z_i) / sum(exp(z_j)):
    ds_i/dz_j = s_i * (delta_ij - s_j)
    where delta_ij is 1 if i==j, else 0
    """
    z = np.array(z).flatten()
    s = softmax(z)
    
    # Create Jacobian matrix
    jacobian = np.diag(s) - np.outer(s, s)
    return jacobian

# Compute Jacobian for a sample
z_sample = np.array([1.0, 2.0, 0.5])
jacobian = softmax_jacobian(z_sample)

print("Input logits:", z_sample)
print("Softmax output:", softmax(z_sample))
print("\nJacobian matrix (derivative):")
print(jacobian)
print("\nNote: Diagonal shows how each output affects itself,")
print("Off-diagonal shows coupling between outputs.")

## Numerical Stability

In [ ]:
# Demonstrate numerical stability issue and solution
large_logits = np.array([1000.0, 1001.0, 999.0])

print("Large logits:", large_logits)

# Naive approach (will overflow)
print("\nNaive softmax approach:")
try:
    exp_vals = np.exp(large_logits)
    print(f"  exp(logits) contains inf: {np.any(np.isinf(exp_vals))}")
    naive_result = exp_vals / np.sum(exp_vals)
    print(f"  Result contains nan: {np.any(np.isnan(naive_result))}")
except Exception as e:
    print(f"  Error: {e}")

# Stable approach (using softmax function)
print("\nStable softmax approach (subtracting max):")
stable_result = softmax(large_logits)
print(f"  Result: {stable_result}")
print(f"  Sum: {np.sum(stable_result)}")
print(f"  Contains inf: {np.any(np.isinf(stable_result))}")
print(f"  Contains nan: {np.any(np.isnan(stable_result))}")

## Summary

**Softmax Activation Function:**
- Converts raw scores (logits) into probability distribution
- Output values sum to 1 and are between 0 and 1
- Used in multi-class classification output layers
- Generalizes sigmoid activation to multiple classes
- Requires numerical stability considerations for large inputs
- Differentiable for use in backpropagation

**Formula**: $\sigma(z)_j = \frac{e^{z_j}}{\sum_{k} e^{z_k}}$

**Key Property**: For each sample, $\sum_{j} \sigma(z)_j = 1$